In [0]:
from pyspark.sql import functions as F

ORIGEM = "workspace.silver.restaurants"
DESTINO = "workspace.gold.dim_restaurantes"

COLUNAS = [
    "restaurant_id",
    "name",
    "amenity",
    "cuisine_raw",
    "categoria_comida",
    "categoria_fonte",
    "bairro",
    "lat",
    "lon",
    "opening_hours",
    "has_opening_hours",
]

silver = spark.table(ORIGEM)

faltantes = sorted(set(COLUNAS) - set(silver.columns))
if faltantes:
    raise ValueError(f"Colunas ausentes na Silver: {faltantes}")

gold = silver.select(*COLUNAS)

total = gold.count()
ids_unicos = gold.select("restaurant_id").distinct().count()

ids_invalidos = gold.filter(
    F.col("restaurant_id").isNull()
    | (F.trim(F.col("restaurant_id").cast("string")) == "")
).count()

assert total > 0, "A Silver está vazia. A Gold não será sobrescrita."
assert ids_invalidos == 0, "Existem identificadores nulos ou vazios."
assert total == ids_unicos, "Existem restaurant_id duplicados."

print("Restaurantes na origem:", total)
print("Identificadores únicos:", ids_unicos)
print("Identificadores inválidos:", ids_invalidos)
print("Validações aprovadas.")

In [0]:
(
    gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(DESTINO)
)

spark.sql(f"""
COMMENT ON TABLE {DESTINO} IS
'Cadastro completo de restaurantes disponíveis no projeto.
Uma linha por restaurant_id.
Use esta tabela para contar restaurantes e analisar categorias,
bairros e disponibilidade de horários.
Não representa apenas as recomendações do momento.
Não contém avaliações de clientes, preços ou tempo de entrega.'
""")

COMENTARIOS = {
    "restaurant_id": "Identificador único do estabelecimento no OpenStreetMap.",
    "name": "Nome do estabelecimento.",
    "amenity": "Tipo de estabelecimento cadastrado no OpenStreetMap.",
    "cuisine_raw": "Informação original de cozinha; pode estar ausente.",
    "categoria_comida": "Categoria atribuída pelas regras do projeto.",
    "categoria_fonte": "Origem da classificação: cuisine, nome, amenity ou default.",
    "bairro": "Bairro cadastrado; pode estar ausente.",
    "lat": "Latitude do estabelecimento.",
    "lon": "Longitude do estabelecimento.",
    "opening_hours": "Horário cadastrado no formato OpenStreetMap; pode estar ausente.",
    "has_opening_hours": "Indica presença de horário cadastrado, não se está aberto agora.",
}

for coluna, descricao in COMENTARIOS.items():
    spark.sql(
        f"ALTER TABLE {DESTINO} "
        f"ALTER COLUMN `{coluna}` COMMENT '{descricao}'"
    )

print("Tabela gravada e documentada:", DESTINO)

In [0]:
resultado = spark.table(DESTINO)

total_gold = resultado.count()
unicos_gold = resultado.select("restaurant_id").distinct().count()

assert total_gold == total, "Quantidade gravada diferente da origem."
assert total_gold == unicos_gold, "Identificadores duplicados na Gold."

print("Total na Silver:", total)
print("Total na Gold:", total_gold)
print("IDs únicos na Gold:", unicos_gold)

display(
    resultado.groupBy("categoria_comida")
    .count()
    .orderBy(F.desc("count"))
)